In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

In [2]:
msl_new = pd.read_csv('../input_data/msl_new.csv')
print(len(msl_new))
msl_new.head()

16648


,Chromophore,Solvent,Quantum yield
0,O=C([O-])c1ccccc1-c1c2ccc(=O)cc-2oc2cc([O-])ccc12,O,0.950
1,O=C([O-])c1ccccc1C1=c2cc3c4c(c2Oc2c1cc1c5c2CCC...,CO,1.000
2,O=C([O-])c1ccccc1-c1c2cc(Br)c(=O)c(Br)c-2oc2c(...,O,0.200
3,O=C([O-])c1ccccc1-c1c2cc(I)c(=O)c(I)c-2oc2c(I)...,O,0.020
4,O=C([O-])c1c(Cl)c(Cl)c(Cl)c(Cl)c1-c1c2cc(I)c(=...,O,0.018


In [3]:
vecs_mols = torch.load('../torch_data/mols_new.pt')
vecs_sols = torch.load('../torch_data/sols_new.pt')

C:\Users\kappa\AppData\Local\Temp\ipykernel_34640\4242151166.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  vecs_mols = torch.load('../torch_data/mols_new.pt')
C:\Users

In [4]:
fgp_mols = torch.load('../torch_data/fingerprints.pt')
fgp_sols = torch.load('../torch_data/fingerprints_sol.pt')
print(len(fgp_mols), len(fgp_sols))
print(fgp_mols.shape, fgp_sols.shape)

C:\Users\kappa\AppData\Local\Temp\ipykernel_34640\42225485.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  fgp_mols = torch.load('../torch_data/fingerprints.pt')
C:\User

16648 16648
torch.Size([16648, 2048]) torch.Size([16648, 2048])


In [5]:
desc_mols = torch.load("../torch_data/descriptors.pt")
desc_sols = torch.load("../torch_data/descriptors_sols.pt")

C:\Users\kappa\AppData\Local\Temp\ipykernel_34640\3135145622.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  desc_mols = torch.load("../torch_data/descriptors.pt")
C:\Us

In [6]:
Xf = torch.cat([fgp_mols, fgp_sols], dim=1)

print(Xf.shape)

yf = torch.tensor(msl_new['Quantum yield'].values)
print(yf.shape)

torch.Size([16648, 4096])
torch.Size([16648])


In [7]:
X_trainf, X_tempf, y_trainf, y_tempf = train_test_split(Xf, yf, test_size=0.2, random_state=42)
X_valf, X_testf, y_valf, y_testf = train_test_split(X_tempf, y_tempf, test_size=0.5, random_state=42)
print(X_trainf.shape, y_trainf.shape)
print(X_valf.shape, y_valf.shape)
print(X_testf.shape, y_testf.shape)

torch.Size([13318, 4096]) torch.Size([13318])
torch.Size([1665, 4096]) torch.Size([1665])
torch.Size([1665, 4096]) torch.Size([1665])


In [8]:
xgb_model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=4,
    reg_lambda=2.0,
    reg_alpha=0.2,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    tree_method="hist",
    device="cuda"
)

eval_set = [(X_valf, y_valf)]

xgb_model.fit(X_trainf, y_trainf, eval_set=eval_set, verbose=False)

y_test_xgb = xgb_model.predict(X_testf)
print("test:")
print(f"R Squared: {r2_score(y_testf, y_test_xgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_testf, y_test_xgb)):.3f}")

test:
R Squared: 0.661
RMSE: 0.176


In [9]:
X = torch.cat([vecs_mols, vecs_sols, desc_mols, desc_sols, fgp_mols, fgp_sols], dim=1)

print(X.shape)

y = torch.tensor(msl_new['Quantum yield'].values)
print(y.shape)

torch.Size([16648, 5042])
torch.Size([16648])


In [10]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

torch.Size([13318, 5042]) torch.Size([13318])
torch.Size([1665, 5042]) torch.Size([1665])
torch.Size([1665, 5042]) torch.Size([1665])


In [11]:
et_model = ExtraTreesRegressor(
    n_estimators=100,
    random_state=42
)

In [12]:
et_model.fit(X_train, y_train)

y_pred_et = et_model.predict(X_test)
print("test:")
print(f"R Squared: {r2_score(y_test, y_pred_et):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_et)):.3f}")

test:
R Squared: 0.722
RMSE: 0.159


### Fingerprint Only

#### XGB

In [6]:
Xf = torch.cat([fgp_mols, fgp_sols], dim=1)

print(Xf.shape)

yf = torch.tensor(msl_new['Quantum yield'].values)
print(yf.shape)

torch.Size([16648, 4096])
torch.Size([16648])


In [7]:
X_trainf, X_tempf, y_trainf, y_tempf = train_test_split(Xf, yf, test_size=0.2, random_state=42)
X_valf, X_testf, y_valf, y_testf = train_test_split(X_tempf, y_tempf, test_size=0.5, random_state=42)
print(X_trainf.shape, y_trainf.shape)
print(X_valf.shape, y_valf.shape)
print(X_testf.shape, y_testf.shape)

torch.Size([13318, 4096]) torch.Size([13318])
torch.Size([1665, 4096]) torch.Size([1665])
torch.Size([1665, 4096]) torch.Size([1665])


In [8]:
xgb_model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=4,
    reg_lambda=2.0,
    reg_alpha=0.2,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    tree_method="hist",
    device="cuda"
)

eval_set = [(X_valf, y_valf)]

xgb_model.fit(X_trainf, y_trainf, eval_set=eval_set, verbose=False)

y_test_xgb = xgb_model.predict(X_testf)
print("test:")
print(f"R Squared: {r2_score(y_testf, y_test_xgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_testf, y_test_xgb)):.3f}")

test:
R Squared: 0.661
RMSE: 0.176


#### Extra Trees

In [10]:
et_model = ExtraTreesRegressor(
    n_estimators=100,
    random_state=42
)

et_model.fit(X_trainf, y_trainf)

y_pred_et = et_model.predict(X_testf)
print("test:")
print(f"R Squared: {r2_score(y_testf, y_pred_et):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_testf, y_pred_et)):.3f}")

test:
R Squared: 0.715
RMSE: 0.161


#### RDKit Only

In [6]:
Xr = torch.cat([desc_mols, desc_sols], dim=1)

print(Xr.shape)

yr = torch.tensor(msl_new['Quantum yield'].values)
print(yr.shape)

torch.Size([16648, 434])
torch.Size([16648])


In [7]:
X_trainr, X_tempr, y_trainr, y_tempr = train_test_split(Xr, yr, test_size=0.2, random_state=42)
X_valr, X_testr, y_valr, y_testr = train_test_split(X_tempr, y_tempr, test_size=0.5, random_state=42)
print(X_trainr.shape, y_trainr.shape)
print(X_valr.shape, y_valr.shape)
print(X_testr.shape, y_testr.shape)

torch.Size([13318, 434]) torch.Size([13318])
torch.Size([1665, 434]) torch.Size([1665])
torch.Size([1665, 434]) torch.Size([1665])


#### XGB

In [10]:
xgb_model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=4,
    reg_lambda=2.0,
    reg_alpha=0.2,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    tree_method="hist",
    device="cuda"
)

eval_set = [(X_valr, y_valr)]

xgb_model.fit(X_trainr, y_trainr, eval_set=eval_set, verbose=False)

y_test_xgb = xgb_model.predict(X_testr)
print("test:")
print(f"R Squared: {r2_score(y_testr, y_test_xgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_testr, y_test_xgb)):.3f}")

test:
R Squared: 0.720
RMSE: 0.159


#### Extra Trees

In [9]:
et_model = ExtraTreesRegressor(
    n_estimators=100,
    random_state=42
)

et_model.fit(X_trainr, y_trainr)

y_pred_et = et_model.predict(X_testr)
print("test:")
print(f"R Squared: {r2_score(y_testr, y_pred_et):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_testr, y_pred_et)):.3f}")

test:
R Squared: 0.714
RMSE: 0.161


### COATI Only

In [6]:
Xc = torch.cat([vecs_mols, vecs_sols], dim=1)

print(Xc.shape)

yc = torch.tensor(msl_new['Quantum yield'].values)
print(yc.shape)

torch.Size([16648, 512])
torch.Size([16648])


In [7]:
X_trainc, X_tempc, y_trainc, y_tempc = train_test_split(Xc, yc, test_size=0.2, random_state=42)
X_valc, X_testc, y_valc, y_testc = train_test_split(X_tempc, y_tempc, test_size=0.5, random_state=42)
print(X_trainc.shape, y_trainc.shape)
print(X_valc.shape, y_valc.shape)
print(X_testc.shape, y_testc.shape)

torch.Size([13318, 512]) torch.Size([13318])
torch.Size([1665, 512]) torch.Size([1665])
torch.Size([1665, 512]) torch.Size([1665])


#### XGB

In [10]:
xgb_model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=4,
    reg_lambda=2.0,
    reg_alpha=0.2,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    tree_method="hist",
    device="cuda"
)

eval_set = [(X_valc, y_valc)]

xgb_model.fit(X_trainc, y_trainc, eval_set=eval_set, verbose=False)

y_test_xgb = xgb_model.predict(X_testc)
print("test:")
print(f"R Squared: {r2_score(y_testc, y_test_xgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_testc, y_test_xgb)):.3f}")

test:
R Squared: 0.682
RMSE: 0.170


#### Extra Trees

In [11]:
et_model = ExtraTreesRegressor(
    n_estimators=100,
    random_state=42
)

et_model.fit(X_trainc, y_trainc)

y_pred_et = et_model.predict(X_testc)
print("test:")
print(f"R Squared: {r2_score(y_testc, y_pred_et):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_testc, y_pred_et)):.3f}")

test:
R Squared: 0.656
RMSE: 0.177


### Fingerprints + RDKit

In [6]:
Xfr = torch.cat([desc_mols, desc_sols, fgp_mols, fgp_sols], dim=1)

print(Xfr.shape)

yfr = torch.tensor(msl_new['Quantum yield'].values)
print(yfr.shape)

torch.Size([16648, 4530])
torch.Size([16648])


In [7]:
X_trainfr, X_tempfr, y_trainfr, y_tempfr = train_test_split(Xfr, yfr, test_size=0.2, random_state=42)
X_valfr, X_testfr, y_valfr, y_testfr = train_test_split(X_tempfr, y_tempfr, test_size=0.5, random_state=42)
print(X_trainfr.shape, y_trainfr.shape)
print(X_valfr.shape, y_valfr.shape)
print(X_testfr.shape, y_testfr.shape)

torch.Size([13318, 4530]) torch.Size([13318])
torch.Size([1665, 4530]) torch.Size([1665])
torch.Size([1665, 4530]) torch.Size([1665])


#### XGB

In [13]:
xgb_model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=4,
    reg_lambda=2.0,
    reg_alpha=0.2,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    tree_method="hist",
    device="cuda"
)

eval_set = [(X_valfr, y_valfr)]

xgb_model.fit(X_trainfr, y_trainfr, eval_set=eval_set, verbose=False)

y_test_xgb = xgb_model.predict(X_testfr)
print("test:")
print(f"R Squared: {r2_score(y_testfr, y_test_xgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_testfr, y_test_xgb)):.3f}")

test:
R Squared: 0.726
RMSE: 0.158


#### Extra Trees

In [14]:
et_model = ExtraTreesRegressor(
    n_estimators=100,
    random_state=42
)

et_model.fit(X_trainfr, y_trainfr)

y_pred_et = et_model.predict(X_testfr)
print("test:")
print(f"R Squared: {r2_score(y_testfr, y_pred_et):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_testfr, y_pred_et)):.3f}")

test:
R Squared: 0.742
RMSE: 0.153


#### 4 COATI (Chromophore) +  SoDaDE (Solvent)

In [11]:
Xfr = torch.cat([vecs_mols, desc_sols, fgp_sols], dim=1)

print(Xfr.shape)

yfr = torch.tensor(msl_new['Quantum yield'].values)
print(yfr.shape)

torch.Size([16648, 2521])
torch.Size([16648])


In [12]:
X_trainfr, X_tempfr, y_trainfr, y_tempfr = train_test_split(Xfr, yfr, test_size=0.2, random_state=42)
X_valfr, X_testfr, y_valfr, y_testfr = train_test_split(X_tempfr, y_tempfr, test_size=0.5, random_state=42)
print(X_trainfr.shape, y_trainfr.shape)
print(X_valfr.shape, y_valfr.shape)
print(X_testfr.shape, y_testfr.shape)

torch.Size([13318, 2521]) torch.Size([13318])
torch.Size([1665, 2521]) torch.Size([1665])
torch.Size([1665, 2521]) torch.Size([1665])


In [13]:
xgb_model = XGBRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=4,
    reg_lambda=2.0,
    reg_alpha=0.2,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50,
    tree_method="hist",
    device="cuda"
)

eval_set = [(X_valfr, y_valfr)]

xgb_model.fit(X_trainfr, y_trainfr, eval_set=eval_set, verbose=False)

y_test_xgb = xgb_model.predict(X_testfr)
print("test:")
print(f"R Squared: {r2_score(y_testfr, y_test_xgb):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_testfr, y_test_xgb)):.3f}")

test:
R Squared: 0.671
RMSE: 0.173


In [14]:
et_model = ExtraTreesRegressor(
    n_estimators=100,
    random_state=42
)

et_model.fit(X_trainfr, y_trainfr)

y_pred_et = et_model.predict(X_testfr)
print("test:")
print(f"R Squared: {r2_score(y_testfr, y_pred_et):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_testfr, y_pred_et)):.3f}")

test:
R Squared: 0.670
RMSE: 0.173
